# PSSN Week 3 — Select eligible Black Hole traces

Run after Week 2 notebook 09. This selects baseline-correct traces with validated states and representations. It stores a full eligibility audit and never modifies source data. See `docs/black_hole_trace_selection.md`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")
from src.black_hole.trace_selection import select_eligible_traces, save_trace_selection
from src.data.semantic_state_dataset import read_jsonl
print("W3-001 selector ready.")

In [ ]:
# Keep None to use all eligible traces; use a positive integer for a reproducible subset.
RANDOM_SEED = 42
SAMPLE_SIZE = None
MIN_STATE_COUNT = 2
MIN_STATE_TOKENS = 3

ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts/expanded_baseline")
candidates = sorted(ARTIFACT_ROOT.rglob("semantic_state_dataset/*/semantic_states.jsonl"))
if not candidates:
    raise FileNotFoundError("No Week 2 semantic_states.jsonl found. Complete notebook 09 first.")
source_path = candidates[-1]
source_records = read_jsonl(source_path)
selected, eligibility_audit, selection_summary = select_eligible_traces(
    source_records, random_seed=RANDOM_SEED, sample_size=SAMPLE_SIZE,
    min_state_count=MIN_STATE_COUNT, min_state_tokens=MIN_STATE_TOKENS,
)
print("Input dataset:", source_path)
print("Summary:", selection_summary)

In [ ]:
from datetime import datetime, timezone
import hashlib, json

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

selection_summary.update({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dataset_path": str(source_path),
    "source_dataset_sha256": sha256_file(source_path),
    "repository_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or None,
    "selector_sha256": sha256_file(Path("src/black_hole/trace_selection.py")),
})
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = source_path.parents[2]
output_dir = run_dir / "black_hole_trace_selection" / stamp
output_paths = save_trace_selection(output_dir, selected, eligibility_audit, selection_summary)
print(json.dumps(selection_summary, indent=2))
print("Saved artifacts:")
for name, path in output_paths.items(): print(f"{name}: {path}")

In [ ]:
saved_selected = read_jsonl(output_paths["selected"])
saved_audit = read_jsonl(output_paths["audit"])
assert len(saved_selected) == selection_summary["selected_trace_count"]
assert len(saved_audit) == selection_summary["input_trace_count"]
assert all(state_id not in [state["state_id"] for state in row["baseline_result"]["states"] if state["is_final_state"]] for row in saved_selected for state_id in row["removable_state_ids"])
print(f"Verified {len(saved_selected)} selected traces; {len(saved_audit)} audited.")